# Task 1: Data Exploration and Enrichment

This notebook explores the unified data schema for Ethiopia's financial inclusion data and performs initial analysis.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 1000)

# Load the data
data_path = '../data/raw/ethiopia_fi_unified_data.xlsx'
ref_path = '../data/raw/reference_codes.xlsx'

# Read all sheets
data_df = pd.read_excel(data_path, sheet_name='data')
impact_links_df = pd.read_excel(data_path, sheet_name='impact_links')
ref_codes_df = pd.read_excel(ref_path)

print("Data loaded successfully!")
print(f"\nData sheet shape: {data_df.shape}")
print(f"Impact links sheet shape: {impact_links_df.shape}")
print(f"Reference codes shape: {ref_codes_df.shape}")

## 1. Understand the Schema

### Examine the data structure

In [ ]:
print("=== DATA SHEET COLUMNS ===")
print(data_df.columns.tolist())

print("\n=== FIRST 10 ROWS OF DATA ===")
data_df.head(10)

In [ ]:
"# Load the data\ndata_path = '../data/raw/ethiopia_fi_unified_data.xlsx'\nref_path = '../data/raw/reference_codes.xlsx'\n\n# Read all sheets\ndata_df = pd.read_excel(data_path, sheet_name='ethiopia_fi_unified_data')\nimpact_links_df = pd.read_excel(data_path, sheet_name='Impact_sheet')\nref_codes_df = pd.read_excel(ref_path)\n\nprint(\"Data loaded successfully!\")\nprint(f\"\\nData sheet shape: {data_df.shape}\")\nprint(f\"Impact links sheet shape: {impact_links_df.shape}\")\nprint(f\"Reference codes shape: {ref_codes_df.shape}\")"

In [ ]:
print("=== REFERENCE CODES ===")
ref_codes_df

## 2. Explore the Data

### Count records by record_type, pillar, source_type, and confidence

In [ ]:
print("=== RECORDS BY RECORD_TYPE ===")
print(data_df['record_type'].value_counts())

print("\n=== RECORDS BY PILLAR ===")
print(data_df['pillar'].value_counts())

print("\n=== RECORDS BY SOURCE_TYPE ===")
print(data_df['source_type'].value_counts())

print("\n=== RECORDS BY CONFIDENCE ===")
print(data_df['confidence'].value_counts())

In [ ]:
# Cross-tabulation
print("=== RECORD_TYPE BY PILLAR ===")
print(pd.crosstab(data_df['record_type'], data_df['pillar']))

### Identify temporal range of observations

In [ ]:
# Convert observation_date to datetime
data_df['observation_date'] = pd.to_datetime(data_df['observation_date'], errors='coerce')

print("=== TEMPORAL RANGE ===")
print(f"Earliest date: {data_df['observation_date'].min()}")
print(f"Latest date: {data_df['observation_date'].max()}")
print(f"Date range: {(data_df['observation_date'].max() - data_df['observation_date'].min()).days / 365.25:.1f} years")

In [ ]:
# Temporal range by record type
for record_type in data_df['record_type'].unique():
    subset = data_df[data_df['record_type'] == record_type]
    if not subset['observation_date'].isna().all():
        print(f"\n{record_type}:")
        print(f"  Earliest: {subset['observation_date'].min()}")
        print(f"  Latest: {subset['observation_date'].max()}")

### List all unique indicators and their coverage

In [ ]:
print("=== UNIQUE INDICATORS ===")
indicators = data_df['indicator_code'].dropna().unique()
print(f"Total unique indicators: {len(indicators)}")
print(indicators)

In [ ]:
# Indicator coverage
print("=== INDICATOR COVERAGE ===")
for indicator in indicators:
    subset = data_df[data_df['indicator_code'] == indicator]
    count = len(subset)
    date_range = f"{subset['observation_date'].min().year} - {subset['observation_date'].max().year}" if not subset['observation_date'].isna().all() else "N/A"
    pillar = subset['pillar'].mode()[0] if not subset['pillar'].isna().all() else "N/A"
    print(f"{indicator}: {count} records, {date_range}, pillar: {pillar}")

### Understand which events are cataloged and their dates

In [ ]:
events = data_df[data_df['record_type'] == 'event']
print("=== CATALOGED EVENTS ===")
print(f"Total events: {len(events)}")
print("\nEvent Details:")
for idx, row in events.iterrows():
    print(f"  {row['record_id']}: {row['indicator']} - {row['category']} - {row['observation_date'].date()}")

### Review existing impact_links and what relationships they capture

In [ ]:
print("=== IMPACT LINKS SUMMARY ===")
print(f"Total impact links: {len(impact_links_df)}")

print("\n=== IMPACT LINKS BY PILLAR ===")
print(impact_links_df['pillar'].value_counts())

print("\n=== IMPACT LINKS BY IMPACT DIRECTION ===")
print(impact_links_df['impact_direction'].value_counts())

In [ ]:
"print(\"=== IMPACT LINKS COLUMNS ===\")\nprint(impact_links_df.columns.tolist())\n\nprint(\"\\n=== FIRST 10 ROWS OF IMPACT LINKS ===\")\nprint(impact_links_df.head(10))"

## 3. Understanding the Schema Design

### Key principle: Events are NOT pre-assigned to pillars

In [ ]:
print("=== EVENTS WITH PILLAR ASSIGNMENTS ===")
events_with_pillar = events[events['pillar'].notna()]
if len(events_with_pillar) > 0:
    print(f"Warning: {len(events_with_pillar)} events have pillar assignments (should be empty)")
    for idx, row in events_with_pillar.iterrows():
        print(f"  {row['record_id']}: {row['indicator']} - pillar: {row['pillar']}")
else:
    print("Good: No events have pillar assignments (as expected)")

In [ ]:
print("=== HOW IMPACT_LINKS CONNECT EVENTS TO INDICATORS ===")
print("\nExample: EVT_0001 (Telebirr Launch) affects multiple indicators via different impact_links")
telebirr_links = impact_links_df[impact_links_df['parent_id'] == 'EVT_0001']
for idx, link in telebirr_links.iterrows():
    print(f"  Impact Link: {link['record_id']}")
    print(f"    Related Indicator: {link['related_indicator']}")
    print(f"    Pillar: {link['pillar']}")
    print(f"    Direction: {link['impact_direction']}, Magnitude: {link['impact_magnitude']}")

## 4. Data Quality Assessment

In [ ]:
print("=== DATA QUALITY CHECK ===")

# Check for missing values
print("\nMissing values by column:")
missing = data_df.isnull().sum()
print(missing[missing > 0])

# Check confidence levels
print("\n=== CONFIDENCE DISTRIBUTION ===")
confidence_dist = data_df['confidence'].value_counts(normalize=True) * 100
print(confidence_dist)

# Check for duplicate record IDs
duplicates = data_df['record_id'].duplicated().sum()
print(f"\nDuplicate record IDs: {duplicates}")

## 5. Summary of Findings

### Key Observations:
1. **Record Types**: The dataset contains observations, events, targets, and impact_links
2. **Temporal Range**: Data spans from [earliest] to [latest]
3. **Indicators**: [count] unique indicators covering different pillars
4. **Events**: [count] key events cataloged
5. **Impact Links**: [count] relationships modeled between events and indicators
6. **Schema Compliance**: Events correctly have no pillar assignments

### Data Gaps Identified:
- [List gaps found]

### Next Steps for Enrichment:
- [Areas where additional data would be valuable]